In [3]:
import os
if 'google.colab' in str(get_ipython()):
    # Running in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks"
else:
    # Running locally (Mac/Linux)
    PROJECT_DIR = '/Users/erikdalgard/Library/CloudStorage/GoogleDrive-dalgard.erik@gmail.com/My Drive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks'

os.chdir(PROJECT_DIR)


In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tqdm
from tensorflow import keras
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, BackupAndRestore
from sklearn.utils.class_weight import compute_class_weight
from datetime import datetime
import glob
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from cub_cutter_c import LunaDataset, make_sequences
from tqdm.auto import tqdm
from froc_eval import evaluate, bootstrap_cpm, plot_froc, compute_froc, froc_table


#Importing modules
import models_2D, models_3D, models_ae

#Changing so keras trains with float16 instead of float32 to increase computational speed.
#mixed_precision.set_global_policy('mixed_float16')

ImportError: cannot import name 'froc_table' from 'froc_eval' (/Users/erikdalgard/Library/CloudStorage/GoogleDrive-dalgard.erik@gmail.com/My Drive/Teknisk fysik/Utbyte/Kurser/DAML/Project/notebooks/froc_eval.py)

In [5]:
#Getting 2D CNN models
archi_1_2D = models_2D.get_archi_1_2D()
archi_2_2D = models_2D.get_archi_2_2D()
archi_3_2D = models_2D.get_archi_3_2D()

#Getting 3D CNN models
archi_1_3D = models_3D.get_archi_1_3D()
archi_2_3D = models_3D.get_archi_2_3D()
archi_3_3D = models_3D.get_archi_3_3D()

#Getting autoencoder models
archi_1_ae = models_ae.get_archi_1_AE()
archi_2_ae = models_ae.get_archi_2_AE()
archi_3_ae = models_ae.get_archi_3_AE()

NameError: name 'models_2D' is not defined

In [ ]:
def train_network(model, X_train, y_train=None, X_val=None, y_val=None, project_dir="", epochs=50, batch_size=64, is_ae=False, is_debug = False):
    """
    Compiles, logs, and trains a given Keras model. Supports both standard classification
    and Autoencoder (AE) reconstruction training.

    Parameters:
        model (keras.Model): The uncompiled Keras model to be trained.
        X_train: Training features.
        y_train: Training labels (Ignored if is_ae=True).
        X_val: Validation features (Optional).
        y_val: Validation labels (Ignored if is_ae=True).
        project_dir (str): Directory for saving assets.
        epochs (int): Maximum number of training iterations.
        batch_size (int): Number of samples per training batch.
        is_ae (bool): Set to True if training an autoencoder.
        is_debug (bool): If set to True, only run 3 epochs on a small set of the data. No data/model is saved

    Returns:
        History: The history as a keras object
        checkpoint_path: The file path to the weights of the model
        log_path: The file path to the csv logs of the training


    """

    #If X_train is a keras.util sequence or a numpy array, determines how we train and load the model
    is_seq = isinstance(X_train, keras.utils.Sequence)


    #If debug is True we only take a small piece of the data to confirm all is working
    if is_debug:
        print("=" * 50)
        print("DEBUG MODE ACTIVE — using subset of data, 3 epochs")
        print("=" * 50)
        if not is_seq:
          X_train = X_train[:128]
          y_train = y_train[:128] if y_train is not None else None
          X_val   = X_val[:32]   if X_val   is not None else None
          y_val   = y_val[:32]   if y_val   is not None else None
        epochs = 3

    # 1. Dynamically configure Loss, Metrics, and Targets based on is_ae flag
    if is_ae:
        print(f"Configuring pipeline for Autoencoder reconstruction task...")
        loss_function = 'mse'
        metrics_list = None  # MSE loss itself acts as the performance tracker for AEs

        # In an autoencoder, the input IS the target output
        train_targets = X_train
        val_targets = X_val if X_val is not None else None
        monitor_metric = 'val_loss'
        monitor_mode = 'min'

        class_weight_dict = None

    else:
        loss_function = 'binary_crossentropy'
        metrics_list = [
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='sensitivity'),
            keras.metrics.AUC(name='auc')
        ]
        train_targets = y_train
        val_targets = y_val
        monitor_metric = 'val_auc'
        monitor_mode = 'max'

        #Calculating class weights to punish positives more than negatives
        y_for_weights = X_train.all_labels if is_seq else y_train
        classes = np.unique(y_for_weights)
        weights = compute_class_weight('balanced', classes=classes, y=y_for_weights.ravel())
        class_weight_dict = dict(zip(classes, weights))

    # 2. Compile the model with the chosen settings
    model.compile(
        optimizer='adam',
        loss=loss_function,
        metrics=metrics_list
    )

    # 3. Create a clean subfolder dynamically named after the architecture
    model_folder = os.path.join(project_dir, 'training_history')
    history_folder = os.path.join(model_folder, model.name)
    os.makedirs(model_folder, exist_ok=True)
    os.makedirs(history_folder, exist_ok=True)

    time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = os.path.join(history_folder, f'best_model_{time_stamp}.keras')
    log_path = os.path.join(history_folder, f'training_log_{time_stamp}.csv')

    # 4. Setting up automation callbacks
    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor_metric,
            mode=monitor_mode,
            save_best_only=True,
            save_weights_only=False  # Explicitly set this to prevent default leaks
        ),
        EarlyStopping( #If model does not improve after 10 epochs, we stop training and restore best model
            monitor=monitor_metric,
            mode=monitor_mode,
            patience=5,
            restore_best_weights=True,
        ),
        CSVLogger(log_path), #logs the training

        ReduceLROnPlateau( #If we stop learning, we wait 5 epochs and then half the learning rate
            monitor=monitor_metric,
            mode=monitor_mode,
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        BackupAndRestore( #If google collabe crashes, this creates a backup folder with the weights of the previous training. If training is succeeded the backup file is deleted.
            backup_dir=os.path.join(history_folder, 'backup')
    ),
    ]

    #No need to save weights and history if we are doing debug.
    if is_debug:
        callbacks = [EarlyStopping(monitor=monitor_metric, mode=monitor_mode, patience=2)]


    if X_val is not None:
        validation_data = X_val if is_seq else (X_val, val_targets) if val_targets is not None else None
    else:
        validation_data = None


    # 5. Execute Training
    print(f"Launching training loop for: {model.name}")


    if is_seq:
        history = model.fit(
            X_train,
            validation_data=validation_data,
            epochs=epochs,
            callbacks=callbacks,
            class_weight=class_weight_dict,
            verbose=1
        )
    else:
        history = model.fit(
            X_train, train_targets,
            validation_data=validation_data,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            class_weight=class_weight_dict,
            verbose=1
        )

    print(f"\nSuccessfully finished training {model.name}!")
    print(f"-> Best weights secured at: {checkpoint_path}")
    print(f"-> History saved to: {log_path}")

    return history, checkpoint_path, log_path

In [ ]:
## OBSERVE THIS IS THE OLD WAY DO NOT USE ANYMORE. Disk → load ALL patches → giant NumPy array in RAM → feed to model. RAM CRASHES
def cut_cubes(arch, layout, subset, is_test=False):
    SUBSET = Path(PROJECT_DIR) / f"data/{subset}"
    VAL_FRACTION = 0.2
    NEG_RATIO = 1
    SEED = 0

    luna = LunaDataset(SUBSET, neg_ratio=NEG_RATIO, seed=SEED, arch=arch, layout=layout)

    feat_shape = tuple(luna.output_signature()[0].shape)
    all_indices = np.arange(len(luna.samples))

    print("feat_shape   :", feat_shape)
    print("total samples:", len(luna.samples))

    def materialize(idxs, desc="Loading samples"):
        X = np.empty((len(idxs), *feat_shape), dtype=np.float32)
        y = np.empty((len(idxs), 1),           dtype=np.float32)
        for k, i in enumerate(tqdm(idxs, desc=desc)):
            X[k], y[k] = luna.get_sample(int(i))
        return X, y

    # --- test mode: no split, return everything as one set ---
    if is_test:
        estimated_gb = (len(all_indices) * np.prod(feat_shape) * 4) / (1024**3)
        print(f"TEST MODE — no train/val split")
        print(f"Estimated size: {estimated_gb:.2f} GB")

        X_test, y_test = materialize(all_indices, desc="Loading test samples")

        print(f"X_test {X_test.shape}  y_test {y_test.shape}  pos {int(y_test.sum())}")
        return X_test, y_test

    # --- normal mode: scan-level train/val split ---
    sample_uids = np.array([
        (luna.pos if label == 1 else luna.neg).iloc[row_i]["seriesuid"]
        for label, row_i, *_ in luna.samples
    ])
    unique  = np.unique(sample_uids)
    n_val   = max(1, int(round(VAL_FRACTION * len(unique))))
    val_uids = set(np.random.default_rng(SEED).choice(unique, size=n_val, replace=False))
    is_val  = np.array([u in val_uids for u in sample_uids])

    estimated_gb = (np.where(~is_val)[0].shape[0] * np.prod(feat_shape) * 4) / (1024**3)
    print(f"Estimated X_train size: {estimated_gb:.2f} GB")

    X_train, y_train = materialize(np.where(~is_val)[0], desc="Loading train samples")
    X_val,   y_val   = materialize(np.where( is_val)[0], desc="Loading val samples")

    print(f"X_train {X_train.shape}  y_train {y_train.shape}  pos {int(y_train.sum())}")
    print(f"X_val   {X_val.shape}    y_val   {y_val.shape}    pos {int(y_val.sum())}")
    return X_train, y_train, X_val, y_val

In [ ]:
#OLD TRAINING SEQUENCE
history, ckpt, log = train_network(
    model=archi_1_3D,          # match ARCH/LAYOUT above
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    project_dir=PROJECT_DIR,
    epochs=50, batch_size=64,
    is_ae=False, is_debug=False,
)


In [ ]:
#NEW LOADING SEQUENCE: Disk → load 64 patches → feed to model → discard → load next 64 patches → .... TALK ABOUT THIS IN PROJECT
class LunaSequence(keras.utils.Sequence):
    """
    Lazy-loading Keras Sequence for LUNA16.
    Loads one batch at a time — no full-dataset materialization needed.
        """
    def __init__(self, luna_dataset, indices, batch_size=64, is_ae=False, shuffle=True):
        super().__init__(workers=4, use_multiprocessing=False)
        self.luna        = luna_dataset
        self.indices     = np.array(indices, dtype=np.int64)
        self.batch_size  = batch_size
        self.is_ae       = is_ae
        self.shuffle     = shuffle
        self._feat_shape = tuple(luna_dataset.output_signature()[0].shape)
        self._labels     = None   # populated lazily for class-weight calculation

    # ── Keras Sequence interface ──────────────────────────────────────────────

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, batch_idx):
        start = batch_idx * self.batch_size
        batch = self.indices[start : start + self.batch_size]

        X = np.empty((len(batch), *self._feat_shape), dtype=np.float32)
        y = np.empty((len(batch), 1),                 dtype=np.float32)
        for k, i in enumerate(batch):
            X[k], y[k] = self.luna.get_sample(int(i))

        # For autoencoders the target is the input itself
        return (X, X) if self.is_ae else (X, y)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)   # re-randomise order each epoch

    # ── Helper ────────────────────────────────────────────────────────────────

    @property
    def all_labels(self):
        """
        Reads labels straight from the sample metadata (no volume I/O).
        Used only for computing class weights in train_network.
        """
        if self._labels is None:
            self._labels = np.array(
                [self.luna.samples[int(i)][0] for i in self.indices],
                dtype=np.float32
            )
        return self._labels

def make_sequences(arch, layout, subset="masked_scans_0-2",
                   val_fraction=0.2, neg_ratio=1, seed=0,
                   batch_size=64, is_ae=False):

    subset_dir = Path(PROJECT_DIR) / f"data/{subset}"
    luna = LunaDataset(subset_dir, neg_ratio=neg_ratio, seed=seed, arch=arch, layout=layout)

    sample_uids = np.array([
        (luna.pos if label == 1 else luna.neg).iloc[row_i]["seriesuid"]
        for label, row_i, *_ in luna.samples
    ])
    unique  = np.unique(sample_uids)
    n_val   = max(1, int(round(val_fraction * len(unique))))
    val_uids = set(np.random.default_rng(seed).choice(unique, size=n_val, replace=False))
    is_val  = np.array([u in val_uids for u in sample_uids])

    train_seq = LunaSequence(luna, np.where(~is_val)[0], batch_size=batch_size, is_ae=is_ae, shuffle=True)
    val_seq   = LunaSequence(luna, np.where( is_val)[0], batch_size=batch_size, is_ae=is_ae, shuffle=False)

    print(f"{arch} {layout} — train batches: {len(train_seq)}, val batches: {len(val_seq)}")
    return train_seq, val_seq

In [ ]:
#FULL TRAINING LOOP
model_list = [
    ("archi1", "2d", archi_1_2D),
    ("archi2", "2d", archi_2_2D),
    ("archi3", "2d", archi_3_2D),
    ("archi1", "3d", archi_1_3D),
    ("archi2", "3d", archi_2_3D),
    ("archi3", "3d", archi_3_3D),
]

for arch, layout, model in model_list:
    train_seq, val_seq = make_sequences(arch, layout, PROJECT_DIR)
    train_network(model, X_train=train_seq, X_val=val_seq, project_dir=PROJECT_DIR)

In [2]:
#LOADING ALL MODELS AFTER TRAINING
model_paths_2D = glob.glob("training_history/2D*/best_model_*.keras", recursive=True)
model_paths_3D = glob.glob("training_history/3D*/best_model_*.keras", recursive=True)
model_paths_AE = glob.glob("training_history/AE*/best_model*.keras", recursive=True)

training_paths_2D = glob.glob("training_history/2D*/training_log*.csv", recursive=True)
training_paths_3D = glob.glob("training_history/3D*/training_log*.csv", recursive=True)
training_paths_AE = glob.glob("training_history/AE*/training_log*.csv", recursive=True)


NameError: name 'glob' is not defined

In [1]:
#TESTING ALL MODELS ON TEST SET

model_paths_2D

NameError: name 'model_paths_2D' is not defined

In [ ]:
#TESTING ALL MODELS ON TEST SET

#Test subset
subset_dir = Path(PROJECT_DIR) / "data/masked_scans3"

#Loading the models
archi_2_2D, archi_1_2D  = keras.models.load_model(model_paths_3D[0]), keras.models.load_model(model_paths_3D[1])

#Assigning weights for linear combination
w1, w2 = 0.43, 0.57
w_total = w1 + w2


#Evaluating the two models
print("Evaluating archi1 2D...")
res1_2D = evaluate(archi_1_2D, subset_dir, arch="archi1", layout="3d")

print("Evaluating archi2 2D...")
res2_2D = evaluate(archi_2_2D, subset_dir, arch="archi2", layout="3d")


ensemble_probs = (w1 * res1_2D["probs"] + w2 * res2_2D["probs"]) / w_total
res_ensemble = compute_froc(ensemble_probs, res1_2D["labels"], res1_2D["n_scans"])

# --- combined plot ---
fig, ax = plt.subplots(figsize=(7, 5))
plot_froc(res1_2D,    ax=ax, label=f"archi1    (CPM={res1_2D['cpm']:.4f})")
plot_froc(res2_2D,    ax=ax, label=f"archi2    (CPM={res2_2D['cpm']:.4f})")
plot_froc(res_ensemble, ax=ax, label=f"Ensemble  (CPM={res_ensemble['cpm']:.4f})")
ax.set_title("FROC comparison — 3D models")
plt.tight_layout()
plt.show()

table = froc_table(
    ("Archi-1",  res1_2D),
    ("Archi-2",  res2_2D),
    ("Ensemble", res_ensemble),
)
print(table)

In [ ]:
table = froc_table(
    ("Archi-1",  res1_2D),
    ("Archi-2",  res2_2D),
    ("Ensemble", res_ensemble),
)
print(table)